# Feature Analysis

Which of the ~340 engineered features carry information about the **next-day return**?

All statistics that select features are computed on the **training period only** (first 80%),
otherwise feature selection itself leaks test information.

In [ ]:
import os, sys
# Find the project root (folder that contains src/) no matter where Jupyter was started
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, 'src')) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
%matplotlib inline
from scipy import stats
from sklearn.feature_selection import mutual_info_regression, f_regression

## 1. Load & engineer features

In [ ]:
from src.pipeline import WARMUP_CALENDAR_DAYS
from src.data_loader import StockDataLoader
from src.preprocess import StockDataPreprocessor, get_feature_columns

SYMBOL, START, END = 'AAPL', '2019-01-01', pd.Timestamp.today().strftime('%Y-%m-%d')
fetch_start = (pd.Timestamp(START) - pd.Timedelta(days=WARMUP_CALENDAR_DAYS)).strftime('%Y-%m-%d')
raw, source = StockDataLoader(SYMBOL).fetch(SYMBOL, fetch_start, END, save=False)
print(f"Source: {source}  rows: {len(raw)}")

df = StockDataPreprocessor(raw, verbose=False).run_pipeline()
df = df[df['Date'] >= START].reset_index(drop=True)
feature_cols = get_feature_columns(df)
TARGET = 'target_return_1d'   # next-day return in %
print(f"Processed: {df.shape}, features: {len(feature_cols)}")

labelled = df.dropna(subset=[TARGET])
split = int(len(labelled) * 0.8)
train = labelled.iloc[:split]
print(f"Training rows used for feature analysis: {len(train)}")

## 2. Feature distributions

In [ ]:
key_features = [f for f in ['Close', 'Volume', 'RSI_14', 'MACD_12_26', 'Volatility_20', TARGET] if f in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, key_features):
    ax.hist(df[col].dropna(), bins=50, alpha=0.7, edgecolor='black')
    ax.set_title(col)
    ax.grid(True, alpha=0.3)
for ax in list(axes.flat)[len(key_features):]:
    ax.remove()
plt.tight_layout(); plt.show()

## 3. Correlation with the target

In [ ]:
correlations = train[feature_cols].corrwith(train[TARGET]).dropna().sort_values()
print("Most negative:"); print(correlations.head(10).round(4))
print("\nMost positive:"); print(correlations.tail(10).round(4))
print(f"\nLargest |corr|: {correlations.abs().max():.3f}  (daily returns are close to unpredictable)")

In [ ]:
top_corr = correlations.abs().sort_values(ascending=False).head(20)
plt.figure(figsize=(10, 7))
plt.barh(top_corr.index[::-1], top_corr.values[::-1], color='steelblue')
plt.xlabel('|correlation| with next-day return'); plt.title('Top 20 features by correlation')
plt.grid(True, alpha=0.3, axis='x'); plt.tight_layout(); plt.show()

top_features = top_corr.index[:15]
plt.figure(figsize=(12, 10))
im = plt.imshow(train[top_features].corr(), cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(top_features)), top_features, rotation=90); plt.yticks(range(len(top_features)), top_features)
plt.colorbar(im); plt.title('Correlation between the top-15 features (redundancy check)')
plt.tight_layout(); plt.show()

## 4. Mutual information (captures non-linear relationships)

In [ ]:
mi_scores = pd.Series(mutual_info_regression(train[feature_cols], train[TARGET], random_state=42),
                      index=feature_cols).sort_values(ascending=False)
print(mi_scores.head(20).round(4))
plt.figure(figsize=(10, 7))
top_mi = mi_scores.head(20)
plt.barh(top_mi.index[::-1], top_mi.values[::-1], color='seagreen')
plt.xlabel('Mutual information'); plt.title('Top 20 features by mutual information')
plt.grid(True, alpha=0.3, axis='x'); plt.tight_layout(); plt.show()

## 5. Scaling: before vs after (fit on train only)

In [ ]:
from sklearn.preprocessing import StandardScaler
sample = [f for f in ['Close', 'Volume', 'RSI_14'] if f in df.columns]
scaler = StandardScaler().fit(train[sample])
scaled = pd.DataFrame(scaler.transform(df[sample]), columns=sample)
fig, axes = plt.subplots(len(sample), 2, figsize=(12, 3 * len(sample)))
for i, col in enumerate(sample):
    axes[i, 0].hist(df[col], bins=50, alpha=0.7, edgecolor='black'); axes[i, 0].set_title(f'{col} - original')
    axes[i, 1].hist(scaled[col], bins=50, alpha=0.7, edgecolor='black', color='orange'); axes[i, 1].set_title(f'{col} - scaled')
plt.tight_layout(); plt.show()

## 6. Custom indicators (backward-looking only)

In [ ]:
df['momentum_10'] = df['Close'].pct_change(10)
df['volume_ratio'] = df['Volume'] / df['Volume'].rolling(20).mean()
low20, high20 = df['Low'].rolling(20).min(), df['High'].rolling(20).max()
df['price_channel'] = (df['Close'] - low20) / (high20 - low20)
df['volatility_ratio'] = df['Volatility_20'] / df['Volatility_20'].rolling(20).mean()

new_features = ['momentum_10', 'volume_ratio', 'price_channel', 'volatility_ratio']
train_new = df.loc[train.index]
print(train_new[new_features].corrwith(train_new[TARGET]).round(4))

fig, axes = plt.subplots(len(new_features), 1, figsize=(15, 10), sharex=True)
for ax, col in zip(axes, new_features):
    ax.plot(df['Date'], df[col], linewidth=1); ax.set_title(col); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Combine selection methods

In [ ]:
f_scores = pd.Series(f_regression(train[feature_cols], train[TARGET])[0], index=feature_cols).fillna(0)
comparison_df = pd.DataFrame({
    'Correlation': correlations.abs().reindex(feature_cols).fillna(0).rank(ascending=False),
    'Mutual_Info': mi_scores.rank(ascending=False),
    'F_Statistic': f_scores.rank(ascending=False),
})
comparison_df['Average_Rank'] = comparison_df.mean(axis=1)
comparison_df = comparison_df.sort_values('Average_Rank')
display(comparison_df.head(20))

## 8. Interaction features

In [ ]:
df['RSI_MACD'] = df['RSI_14'] * df['MACD_12_26']
df['Volume_Volatility'] = df['Volume'] * df['Volatility_20']
df['Price_RSI'] = df['Close'] * df['RSI_14']
inter = ['RSI_MACD', 'Volume_Volatility', 'Price_RSI']
train_new = df.loc[train.index]
print(train_new[inter].corrwith(train_new[TARGET]).round(4))

## 9. Normality tests

In [ ]:
for feature in [f for f in ['Close', 'Volume', 'RSI_14', TARGET] if f in df.columns]:
    data = df[feature].dropna()
    stat, p = stats.shapiro(data.sample(min(5000, len(data)), random_state=42))
    print(f"{feature:20s}: p = {p:.4g} -> {'normal' if p > 0.05 else 'NOT normal'}")

## 10. Save the selected features

In [ ]:
from utils.config import DATA_PROCESSED_DIR
best_features = comparison_df.head(20).index.tolist()
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
path = os.path.join(DATA_PROCESSED_DIR, f'{SYMBOL}_best_features.txt')
with open(path, 'w') as f:
    f.write('\n'.join(best_features))
print(f"✅ Saved {len(best_features)} features to {path}")

## Notes
- Correlations of single features with next-day returns are tiny (|r| < 0.1 is typical).
- Price-level features (SMA, EMA, Close) dominate *price* prediction only because tomorrow's price ≈ today's price; for *returns* they carry little information.
- Validate any selected subset with the time-series CV in `03_experiments.ipynb`.